In [2]:
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda")

SMOKE = False          # set to False for the real run
MAX_LEN, BATCH, ACCUM, EPOCHS, LR = 512, 8, 2, 3, 2e-5

def clean_code(code):
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)
    code = re.sub(r"//[^\n]*", " ", code)
    code = re.sub(r"\s+", " ", code)
    return code.strip()

train = pd.read_json("../data/train.jsonl", lines=True)
valid = pd.read_json("../data/valid.jsonl", lines=True)
if SMOKE:
    train, valid, EPOCHS = train.sample(400, random_state=42), valid.sample(200, random_state=42), 1
train["clean"] = train["func"].apply(clean_code)
valid["clean"] = valid["func"].apply(clean_code)

tok = AutoTokenizer.from_pretrained("microsoft/codebert-base")

def make_loader(df, shuffle):
    items = list(zip(df["clean"], df["target"]))
    def collate(batch):
        texts, labels = zip(*batch)
        enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(labels)
        return enc
    return DataLoader(items, batch_size=BATCH, shuffle=shuffle, collate_fn=collate)

train_dl, valid_dl = make_loader(train, True), make_loader(valid, False)

model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2).to(device)

counts = np.bincount(train["target"].values)
weights = torch.tensor(len(train) / (2 * counts), dtype=torch.float32, device=device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * len(train_dl) // ACCUM
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler("cuda")

def evaluate():
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in valid_dl:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.amp.autocast("cuda"):
                logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
            preds.append(logits.argmax(1).cpu().numpy())
            labels.append(b["labels"].cpu().numpy())
    return np.concatenate(labels), np.concatenate(preds)

best_f1 = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    for i, b in enumerate(train_dl):
        b = {k: v.to(device) for k, v in b.items()}
        with torch.amp.autocast("cuda"):
            logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
            loss = loss_fn(logits.float(), b["labels"]) / ACCUM
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
            sched.step()
        if i % 200 == 0:
            print(f"epoch {epoch} step {i}/{len(train_dl)} loss {loss.item() * ACCUM:.3f}")
    y, p = evaluate()
    acc, f1 = accuracy_score(y, p), f1_score(y, p)
    print(f"epoch {epoch}: val acc={acc:.3f}, F1(vuln)={f1:.3f}")
    if not SMOKE:
        torch.save(model.state_dict(), f"../models/codebert_epoch{epoch}.pt")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1 step 0/2732 loss 0.856
epoch 1 step 200/2732 loss 0.708
epoch 1 step 400/2732 loss 0.660
epoch 1 step 600/2732 loss 0.706
epoch 1 step 800/2732 loss 0.717
epoch 1 step 1000/2732 loss 0.700
epoch 1 step 1200/2732 loss 0.839
epoch 1 step 1400/2732 loss 0.665
epoch 1 step 1600/2732 loss 0.714
epoch 1 step 1800/2732 loss 0.744
epoch 1 step 2000/2732 loss 0.675
epoch 1 step 2200/2732 loss 0.733
epoch 1 step 2400/2732 loss 0.622
epoch 1 step 2600/2732 loss 0.532
epoch 1: val acc=0.574, F1(vuln)=0.637
epoch 2 step 0/2732 loss 0.534
epoch 2 step 200/2732 loss 0.644
epoch 2 step 400/2732 loss 0.577
epoch 2 step 600/2732 loss 0.725
epoch 2 step 800/2732 loss 0.498
epoch 2 step 1000/2732 loss 0.562
epoch 2 step 1200/2732 loss 0.548
epoch 2 step 1400/2732 loss 0.497
epoch 2 step 1600/2732 loss 0.378
epoch 2 step 1800/2732 loss 0.534
epoch 2 step 2000/2732 loss 0.471
epoch 2 step 2200/2732 loss 0.299
epoch 2 step 2400/2732 loss 0.537
epoch 2 step 2600/2732 loss 0.608
epoch 2: val acc=0.638,

In [3]:
from sklearn.metrics import classification_report, confusion_matrix

torch.save(model.state_dict(), "../models/codebert_final.pt")   # epoch-3 weights

test = pd.read_json("../data/test.jsonl", lines=True)
test["clean"] = test["func"].apply(clean_code)
test_dl = make_loader(test, False)

model.eval()
preds, labels = [], []
with torch.no_grad():
    for b in test_dl:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.amp.autocast("cuda"):
            logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
        preds.append(logits.argmax(1).cpu().numpy())
        labels.append(b["labels"].cpu().numpy())
y_test, p_test = np.concatenate(labels), np.concatenate(preds)

print(classification_report(y_test, p_test, digits=3))
print(confusion_matrix(y_test, p_test))

              precision    recall  f1-score   support

           0      0.688     0.627     0.656      1486
           1      0.597     0.661     0.627      1246

    accuracy                          0.642      2732
   macro avg      0.642     0.644     0.641      2732
weighted avg      0.646     0.642     0.643      2732

[[931 555]
 [423 823]]
